<a href="https://colab.research.google.com/github/M-AyyanAsif/NETSOL_TASKS/blob/main/N_Gram_Vanity_Plate_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import math
import random
from collections import defaultdict
import pandas as pd

In [2]:
DATA_FILE = "/content/license_plate_ngram_dataset.csv"
N_ORDER = 3                # Trigram
CONTEXT_LEN = N_ORDER - 1  # 2 previous characters used as context
PAD = "^"                  # Padding symbol for sequence starts
NUM_CANDIDATES = 20        # Candidates generated per seed
TOP_K = 5

In [3]:
df = pd.read_csv(DATA_FILE)
df.head(7)

,Plate_ID,Plate_Text,Letter_Segment,Digit_Segment,Type
0,1,SRG-1824,SRG,1824,Standard
1,2,DGK-4012,DGK,4012,Standard
2,3,MTN-1679,MTN,1679,Custom
3,4,ZAB-1424,ZAB,1424,VIP
4,5,RZA-0488,RZA,488,Standard
5,6,HYD-8279,HYD,8279,VIP
6,7,NBS-3257,NBS,3257,Custom


In [4]:
# STEP 1 & 2: Training Data & Preprocessing
# ==========================================
def load_and_preprocess_data(csv_path):
    # Load raw data and split into per-segment training sets (letters vs digits)
    df = pd.read_csv(csv_path, dtype=str)
    letters = df["Letter_Segment"].str.strip().str.upper().tolist()
    digits = df["Digit_Segment"].str.strip().str.zfill(4).tolist()
    return letters, digits

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Plate_ID        1000 non-null   int64 
 1   Plate_Text      1000 non-null   object
 2   Letter_Segment  1000 non-null   object
 3   Digit_Segment   1000 non-null   int64 
 4   Type            1000 non-null   object
dtypes: int64(2), object(3)
memory usage: 39.2+ KB


In [6]:
# STEP 3: N-Gram Model (Training & Probabilities)
# ==========================================
def train_ngram_model(sequences):
    # Counts character transitions: context -> next_char
    counts = defaultdict(lambda: defaultdict(int))
    vocab = set()

    for seq in sequences:
        seq = str(seq).strip().upper()
        if not seq:
            continue

        vocab.update(seq)
        padded = (PAD * CONTEXT_LEN) + seq

        # Slide window to count occurrences
        for i in range(len(seq)):
            context = padded[i:i + CONTEXT_LEN]
            next_char = padded[i + CONTEXT_LEN]
            counts[context][next_char] += 1

    vocab.add(PAD)
    return counts, vocab

def get_smoothed_prob(context, char, counts, vocab):
    # Laplace (+1) Smoothing
    ctx_counts = counts[context]
    total_seen = sum(ctx_counts.values())
    vocab_size = max(len(vocab), 1)
    return (ctx_counts.get(char, 0) + 1) / (total_seen + vocab_size)

In [7]:
# STEP 4: Generation Pipeline
# ==========================================
def generate_sequence(length, seed, counts, vocab):
    # Force seed to match target length limitations
    seed = str(seed).strip().upper()[:length]
    generated = seed
    padded = (PAD * CONTEXT_LEN) + generated
    vocab_list = list(vocab)

    while len(generated) < length:
        # Get context of previous characters
        ctx = padded[-CONTEXT_LEN:] if CONTEXT_LEN > 0 else ""

        # Sample next character based on N-gram probabilities
        weights = [get_smoothed_prob(ctx, c, counts, vocab) for c in vocab_list]
        next_char = random.choices(vocab_list, weights=weights, k=1)[0]

        if next_char == PAD:
            continue

        generated += next_char
        padded += next_char

    return generated[:length]

In [8]:
# STEP 5 & 6: Validation and Scoring
# ==========================================
def is_valid_schema(plate_text):
    # Must strictly match format: LLL-DDDD
    return bool(re.match(r"^[A-Z]{3}-\d{4}$", plate_text))

def get_sequence_score(seq, counts, vocab):
    # Calculate log-probability to score how natural the sequence looks
    seq = str(seq).strip().upper()
    padded = (PAD * CONTEXT_LEN) + seq
    log_prob = 0.0

    for i in range(len(seq)):
        ctx = padded[i:i + CONTEXT_LEN]
        nxt = padded[i + CONTEXT_LEN]
        log_prob += math.log(get_smoothed_prob(ctx, nxt, counts, vocab))

    return log_prob

In [9]:
# STEP 7 & OUTPUT: Ranking System
# ==========================================
def main():
    print("Loading data...")
    letters_data, digits_data = load_and_preprocess_data(DATA_FILE)

    # Train separate models for separate segments
    letter_counts, letter_vocab = train_ngram_model(letters_data)
    digit_counts, digit_vocab = train_ngram_model(digits_data)

    # Get user seed
    seed = input("Enter a seed (e.g., 'ALI') or press Enter for default: ").strip() or "ALI"
    seed = seed.upper()

    print(f"\nGenerating candidates for '{seed}'...")
    candidates = []

    # Generate candidates
    for _ in range(NUM_CANDIDATES):
        if seed.isalpha():
            l_seq = generate_sequence(3, seed, letter_counts, letter_vocab)
            d_seq = generate_sequence(4, "", digit_counts, digit_vocab)
        elif seed.isdigit():
            l_seq = generate_sequence(3, "", letter_counts, letter_vocab)
            d_seq = generate_sequence(4, seed, digit_counts, digit_vocab)
        else:
            l_seq = generate_sequence(3, "", letter_counts, letter_vocab)
            d_seq = generate_sequence(4, "", digit_counts, digit_vocab)

        plate = f"{l_seq}-{d_seq}"

        # Validate schema
        if is_valid_schema(plate):
            # Score survivor
            score = get_sequence_score(l_seq, letter_counts, letter_vocab) + \
                    get_sequence_score(d_seq, digit_counts, digit_vocab)
            candidates.append((plate, score))

    # Deduplicate and sort by highest score
    unique_plates = {}
    for plate, score in candidates:
        if plate not in unique_plates or score > unique_plates[plate]:
            unique_plates[plate] = score

    ranked = sorted(unique_plates.items(), key=lambda x: x[1], reverse=True)[:TOP_K]

    # Output recommended plates
    print(f"\nTop {TOP_K} Plates:")
    print("-" * 30)
    for rank, (plate, score) in enumerate(ranked, 1):
        print(f"{rank}. {plate}   (Score: {score:.4f})")

if __name__ == "__main__":
    main()

Loading data...
Enter a seed (e.g., 'ALI') or press Enter for default: 56

Generating candidates for '56'...

Top 5 Plates:
------------------------------
1. FAZ-5654   (Score: -13.5308)
2. JHL-5651   (Score: -13.6258)
3. BHN-5662   (Score: -13.9608)
4. TAJ-5626   (Score: -14.0147)
5. AKH-5631   (Score: -14.0457)
